In [209]:
import plotly.io as pio
pio.renderers.default = 'iframe'
'''
from juliacall import Main as jl
jl.seval("import Pkg")
jl.Pkg.add("Ripserer")
jl.Pkg.add("DelimitedFiles")
jl.Pkg.add("DataFrames")
jl.Pkg.add("CSV")
jl.Pkg.add("JSON")
jl.Pkg.add("PersistenceDiagrams")
jl.Pkg.add("StatsBase")
jl.Pkg.add("Suppressor")
jl.include("callumPH.jl")
jl.include("hypergraph_utils.jl")
jl.include("PH2hypergraph.jl")
jl.include("PH_hypergraph_centralities.jl")
jl.include("centrality_tools.jl")
jl.include("ph_to_json.jl")
'''
import os
import numpy as np
import json
import gudhi.wasserstein
from gudhi.wasserstein.barycenter import lagrangian_barycenter as bary
from gudhi.persistence_graphical_tools import plot_persistence_diagram
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

In [2]:
#From Agnese - loads the json files with PH information
def load_PH(name):
    with open(name, 'r') as f:
        dic = json.load(f)
    barcode = dic['barcode']
    repre = dic['representatives']
    repre = [np.unique(np.concatenate(r)) for r in repre]
    return np.array(barcode).T, repre  

In [3]:
#As per Reani and Bobrowski - returns a cycle affinity score (ish) 
def cycle_affinity_measure(int1,int2):
    return max((min(int1[1],int2[1]) - max(int1[0],int2[0])),0)/(max(int1[1],int2[1]) - min(int1[0],int2[0]))

In [4]:
#From Agnese - plots a given cycle onto a given figure of a protein backbone
def plot_cycle(data, cycle, fig, color, label):
    """
    Add a cycle on the existing figure as a CLOSED polyline (and optional markers).
    - cycle: list of 1-based indices into 'data' defining the cycle order.
    - close=True ensures last->first connection.
    - show_markers controls whether to show points as markers along the cycle.
    Transparent background preserved.
    """
    # Convert 1-based indices to 0-based and map to points
    idxs = [a - 1 for a in cycle]
    pts = [data[a] for a in idxs]
    df = pd.DataFrame.from_records(pts, columns=['X', 'Y', 'Z'])

    # Close the cycle by appending first point at the end
    df = pd.concat([df, df.iloc[[0]]], ignore_index=True)


    fig.add_trace(go.Scatter3d(
        x=df['X'], y=df['Y'], z=df['Z'],
        name=label,
        mode='markers',
        marker=dict(size=8, color=color)
    ))

    # Re-assert transparent layout (safe to call again)
    fig.update_layout(
        scene=dict(
            xaxis=dict(backgroundcolor='rgba(0,0,0,0)', gridcolor='rgba(0,0,0,0)',
                       showbackground=False, zerolinecolor='rgba(0,0,0,0)', showticklabels=False),
            yaxis=dict(backgroundcolor='rgba(0,0,0,0)', gridcolor='rgba(0,0,0,0)',
                       showbackground=False, zerolinecolor='rgba(0,0,0,0)', showticklabels=False),
            zaxis=dict(backgroundcolor='rgba(0,0,0,0)', gridcolor='rgba(0,0,0,0)',
                       showbackground=False, zerolinecolor='rgba(0,0,0,0)', showticklabels=False),
            bgcolor='rgba(0,0,0,0)'
        ),
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)'
    ).update_layout(scene=dict(xaxis_title=' ', yaxis_title=' ', zaxis_title=' '))

    return fig

In [5]:
#From Agnese - plots the backbone of a protein
def plot(data):

    df = pd.DataFrame.from_records(data, columns=['X', 'Y', 'Z'])

    
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(
        x=df['X'], y=df['Y'], z=df['Z'],
        name='Knot',
        mode='lines',
        marker=dict(size=8, color='grey'),
        line=dict(width=8, color='grey')
    ))

    # Transparent scene & figure
    fig.update_layout(
        scene=dict(
            xaxis=dict(backgroundcolor='rgba(0,0,0,0)', gridcolor='rgba(0,0,0,0)',
                       showbackground=False, zerolinecolor='rgba(0,0,0,0)', showticklabels=False),
            yaxis=dict(backgroundcolor='rgba(0,0,0,0)', gridcolor='rgba(0,0,0,0)',
                       showbackground=False, zerolinecolor='rgba(0,0,0,0)', showticklabels=False),
            zaxis=dict(backgroundcolor='rgba(0,0,0,0)', gridcolor='rgba(0,0,0,0)',
                       showbackground=False, zerolinecolor='rgba(0,0,0,0)', showticklabels=False),
            bgcolor='rgba(0,0,0,0)'
        ),
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        width = 1000,
        height = 1000
    ).update_layout(scene=dict(xaxis_title=' ', yaxis_title=' ', zaxis_title=' '))

    return fig

In [6]:
#Reads the backbone information from a .xyz file
def load_curve2(name):
    df = pd.read_csv(name, delimiter=' ',header=None)
    return df.values

In [7]:
#Checks which AOTC proteins were successfully processed
successfully_processed_aotc = []
for i in os.listdir("AOTCASE")[1:]:
    if os.path.isfile("AOTCASE/"+i+"/"+i+".json"):
        successfully_processed_aotc.append(i)

In [8]:
#Checks which OTC proteins were successfully processed
successfully_processed_otc = []
for i in os.listdir("OTCASE3")[1:]:
    if os.path.isfile("OTCASE3/"+i+"/"+i+".json"):
        successfully_processed_otc.append(i)

In [13]:
#Retrieves the persistent diagrams and representative cycles for the first 50 AOTCASE proteins
folder = "AOTCASE"
Dgmsk = []
Repsk = []
for code in successfully_processed_aotc[0:50]:
    dgms, representatives = load_PH(f'{folder}/{code}/{code}.json')
    Dgmsk.append(dgms)
    Repsk.append(representatives)

In [14]:
#Retrieves the persistent diagrams and representative cycles for the first 50 OTCASE proteins
folder = "OTCASE3"
Dgmsu = []
Repsu = []
for code in successfully_processed_otc[0:50]:
    dgms, representatives = load_PH(f'{folder}/{code}/{code}.json')
    Dgmsu.append(dgms)
    Repsu.append(representatives)

In [15]:
#Computes the barycentre diagram for the AOTCASE proteins
bk, logk = bary(Dgmsk, 
         init=0,
         verbose=True)

In [16]:
#Computes the barycentre diagram for the OTCASE proteins
bu, logu = bary(Dgmsu, 
         init=0,
         verbose=True)

In [28]:
Gk = logk["groupings"]
Gu = logu["groupings"]

In [162]:
#Computes the affinities for AOTCASE barycentre features
matched_features_affinitiesk = []
for i in bk:
    matched_features_affinitiesk.append([])

for i in range(len(Gk)):
    for match in Gk[i]:
        prot_cycle = match[1]
        bary_cycle = match[0]
        if prot_cycle != -1 and bary_cycle != -1:
            matched_features_affinitiesk[bary_cycle].append(cycle_affinity_measure(bk[bary_cycle],Dgmsk[i][prot_cycle]))
        elif bary_cycle != -1 and prot_cycle == -1:
            matched_features_affinitiesk[bary_cycle].append(0)

In [182]:
#Computes the affinities for AOTCASE barycentre features
matched_features_affinitiesu = []
for i in bu:
    matched_features_affinitiesu.append([])

for i in range(len(Gu)):
    for match in Gu[i]:
        prot_cycle = match[1]
        bary_cycle = match[0]
        if prot_cycle != -1 and bary_cycle != -1:
            matched_features_affinitiesu[bary_cycle].append(cycle_affinity_measure(bu[bary_cycle],Dgmsu[i][prot_cycle]))
        elif bary_cycle != -1 and prot_cycle == -1:
            matched_features_affinitiesu[bary_cycle].append(0)

In [183]:
#Computes the "prevalence scores" for each barycentre feature
cycle_prevalence_scores_k = [(i, sum(matched_features_affinitiesk[i])/len(matched_features_affinitiesk[i])) for i in range(len(matched_features_affinitiesk))]
cycle_prevalence_scores_u = [(i, sum(matched_features_affinitiesu[i])/len(matched_features_affinitiesu[i])) for i in range(len(matched_features_affinitiesu))]

In [184]:
#Also includes the persistence of each feature
cycles_with_prevalence_and_persistence_k = [(cycle[0], cycle[1], bk[cycle[0]][1]-bk[cycle[0]][0]) for cycle in cycle_prevalence_scores_k]
cycles_with_prevalence_and_persistence_u = [(cycle[0], cycle[1], bu[cycle[0]][1]-bu[cycle[0]][0]) for cycle in cycle_prevalence_scores_u]

In [205]:
#Removes low affinity features and sorts by persistence
filtered_cycles_with_prevalence_and_persistence_k = [cycle for cycle in cycles_with_prevalence_and_persistence_k if cycle[1] >= 0.9]
sorted_filtered_cycles_by_persistence_k = sorted(filtered_cycles_with_prevalence_and_persistence_k, key=lambda x: (x[2],x[1]), reverse=True)
filtered_cycles_with_prevalence_and_persistence_u = [cycle for cycle in cycles_with_prevalence_and_persistence_u if cycle[1] >= 0.9]
sorted_filtered_cycles_by_persistence_u = sorted(filtered_cycles_with_prevalence_and_persistence_u, key=lambda x: (x[2],x[1]), reverse=True)

In [206]:
#Attempts to take the representatives in the nth AOTCASE proteim of the top 5 persistent features in the barycentre after removing low affinity features
folder = "AOTCASE"
n = 0 #Select which protein in successfully_processed_aotc to look at

top_5_features_k = [cycle[0] for cycle in sorted_filtered_cycles_by_persistence_k[0:5]]
representing_cycles_k = []
for match in Gk[n]:
    if match[0] in top_5_features_k and match[1] != -1:
        representing_cycles_k.append(match[1])
        
code = successfully_processed_aotc[n]
ck = load_curve2(f'{folder}/{code}/{code}.xyz')

In [207]:
#Plots the representative cycles
figk = plot(ck)
colours = ['red','maroon','orange','green','blue']
for i in range(len(representing_cycles_k)):
    plot_cycle(ck,Repsk[n][representing_cycles_k[i]],figk,colours[i],'cycle')
figk

In [202]:
#Attempts to take the representatives in the nth AOTCASE proteim of the top 5 persistent features in the barycentre after removing low affinity features
folder = "OTCASE3"
n = 0 #Select which protein in successfully_processed_otc to look at

top_5_features_u = [cycle[0] for cycle in sorted_filtered_cycles_by_persistence_u[0:5]]
representing_cycles_u = []
for match in Gu[n]:
    if match[0] in top_5_features_u and match[1] != -1:
        representing_cycles_u.append(match[1])
        
code = successfully_processed_otc[n]
cu = load_curve2(f'{folder}/{code}/{code}.xyz')

In [208]:
#Plots the representative cycles
figu = plot(cu)
colours = ['red','maroon','orange','green','blue']
for i in range(len(representing_cycles_u)):
    plot_cycle(cu,Repsu[n][representing_cycles_u[i]],figu,colours[i],'cycle')
figu